'''
=================================================
MILESTONE 1

Nama  : Aldi Zulkarnaen Alamsyah
Batch : CODA-RMT-023

Program ini dibuat untuk melakukan web scraping data saham paling aktif (Most Active Stocks) dari Yahoo Finance, kemudian melakukan transformasi (cleaning) agar data siap diproses.
=================================================
'''

A. PROSES EKSTRAK DATA DARI YAHOO FINANCE

In [1]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time
import pandas as pd

In [2]:
options = webdriver.ChromeOptions()
options.add_argument('--disable-notifications')
options.page_load_strategy = 'eager'
driver = webdriver.Chrome(options=options)
driver.get('https://finance.yahoo.com/markets/stocks/most-active/?start=0&count=100')
time.sleep(5)
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(5)

soup = BeautifulSoup(driver.page_source, 'html.parser')
most_active = soup.find('table')

In [3]:
driver.quit()

In [4]:
print(most_active)

In [5]:
data_mentah = []
rows = most_active.find_all('tr')
for row in rows[1:]:
    cols = row.find_all('td')

    if len(cols) >= 9:
        raw_price = cols[3].text.strip()
        clean_price = raw_price.split()[0]
        data_mentah.append({
            'Symbol': cols[0].text.strip(),
            'Company Name': cols[1].text.strip(),
            'Price': clean_price,
            'Change Percent': cols[5].text.strip(),
            'Volume': cols[6].text.strip(),
            'Market Cap': cols[8].text.strip()
        })

In [6]:
df_mentah = pd.DataFrame(data_mentah)

In [7]:
display(df_mentah.head())

,Symbol,Company Name,Price,Change Percent,Volume,Market Cap
0,GRAB,Grab Holdings Limited,3.16,+8.42%,151.251M,12.908B
1,NOK,Nokia Oyj,10.77,-1.60%,83.901M,60.325B
2,SOFI,"SoFi Technologies, Inc.",17.11,+0.85%,81.034M,22.073B
3,INTC,Intel Corporation,122.67,+0.73%,68.971M,648.753B
4,WBD,"Warner Bros. Discovery, Inc.",30.81,+0.02%,67.980M,77.245B


In [8]:
print(f'Total data: {df_mentah.shape[0]}')
display(df_mentah.head())

Total data: 100


,Symbol,Company Name,Price,Change Percent,Volume,Market Cap
0,GRAB,Grab Holdings Limited,3.16,+8.42%,151.251M,12.908B
1,NOK,Nokia Oyj,10.77,-1.60%,83.901M,60.325B
2,SOFI,"SoFi Technologies, Inc.",17.11,+0.85%,81.034M,22.073B
3,INTC,Intel Corporation,122.67,+0.73%,68.971M,648.753B
4,WBD,"Warner Bros. Discovery, Inc.",30.81,+0.02%,67.980M,77.245B


In [9]:
df_mentah.to_csv('kotor_most_active.csv', index=False)

B. TRANSFORM : MENGOLAH DATA KOTOR
Pada tahap ini merupakan tahap membersihkan data mentah yang telah diolah sebelumnya.
Target data yang ingin dibersihkan terdapat pada kolom Price, Change Percent, Volume, dan Market Cap.

In [10]:
import pandas as pd
import numpy as np
df = pd.read_csv('kotor_most_active.csv')

In [11]:
df

,Symbol,Company Name,Price,Change Percent,Volume,Market Cap
0,GRAB,Grab Holdings Limited,3.16,+8.42%,151.251M,12.908B
1,NOK,Nokia Oyj,10.77,-1.60%,83.901M,60.325B
2,SOFI,"SoFi Technologies, Inc.",17.11,+0.85%,81.034M,22.073B
3,INTC,Intel Corporation,122.67,+0.73%,68.971M,648.753B
4,WBD,"Warner Bros. Discovery, Inc.",30.81,+0.02%,67.980M,77.245B
...,...,...,...,...,...,...
95,SCHW,The Charles Schwab Corporation,101.39,-5.14%,11.893M,175.329B
96,SBET,"Sharplink, Inc.",10.20,+2.20%,11.785M,2.216B
97,BTE,Baytex Energy Corp.,4.62,-1.07%,11.784M,3.23B
98,AVGO,Broadcom Inc.,364.37,+0.47%,11.531M,1.739T


In [12]:
# Function untuk mengubah nilai dengan suffix T, B, M menjadi angka float
def convert_suffix(value):
    ''' Mengonversi nilai dengan suffix T, B, M menjadi angka float 
    Value str/float adalah nilai dari data yang akan dikonversi
    Return float dari nilai yang dikonversi'''
    if pd.isna(value) or value == '-':
        return 0.0
    value = str(value).replace(',', '')
    if 'T' in value:
        return float(value.replace('T', '')) * 1_000_000_000_000
    elif 'B' in value:
        return float(value.replace('B', '')) * 1_000_000_000
    elif 'M' in value:
        return float(value.replace('M', '')) * 1_000_000
    else:
        return float(value)
# Menerapkan function konversi ke kolom Volume dan Market Cap
df['Volume'] = df['Volume'].apply(convert_suffix)
df['Market Cap'] = df['Market Cap'].apply(convert_suffix)
# Membersihkan kolom change percent dan price agar bisa diubah menjadi float
df['Change Percent'] = df['Change Percent'].str.replace('+', '', regex=False).str.replace('%', '', regex=False)
df['Price'] = df['Price'].astype(str).str.replace(',', '', regex=False).astype(float)
df['Change Percent'] = df['Change Percent'].astype(float)  


BERIKUT MERUPAKAN EKSPLORASI DATA SEDERHANA
setelah membersihkan data, melakukan pengecekan data untuk memastikan tidak ada
df.info untuk validasi bahwa tipe data kolom Price, Change Percent, Volume, dan Market Cap sudah menjadi float
df.describe untuk melihat ringkasan statistik dari data most active stocks

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Symbol          100 non-null    str    
 1   Company Name    100 non-null    str    
 2   Price           100 non-null    float64
 3   Change Percent  100 non-null    float64
 4   Volume          100 non-null    float64
 5   Market Cap      100 non-null    float64
dtypes: float64(4), str(2)
memory usage: 4.8 KB


In [14]:
df.describe()

,Price,Change Percent,Volume,Market Cap
count,100.00000,100.000000,1.000000e+02,1.000000e+02
mean,84.84140,1.340400,2.422384e+07,4.193523e+11
std,166.63652,4.586898,1.896339e+07,1.078040e+12
min,2.10000,-5.260000,1.143800e+07,2.204000e+09
25%,9.72750,-0.392500,1.439525e+07,7.282750e+09
50%,17.94500,0.565000,1.946850e+07,2.196400e+10
75%,70.17750,2.155000,2.409050e+07,1.463512e+11
max,1087.73000,32.200000,1.512510e+08,5.542000e+12


In [15]:
df.head(5)

,Symbol,Company Name,Price,Change Percent,Volume,Market Cap
0,GRAB,Grab Holdings Limited,3.16,8.42,151251000.0,1.290800e+10
1,NOK,Nokia Oyj,10.77,-1.60,83901000.0,6.032500e+10
2,SOFI,"SoFi Technologies, Inc.",17.11,0.85,81034000.0,2.207300e+10
3,INTC,Intel Corporation,122.67,0.73,68971000.0,6.487530e+11
4,WBD,"Warner Bros. Discovery, Inc.",30.81,0.02,67980000.0,7.724500e+10


TAHAP AKHIR DI SINI MERUPAKAN EKSPORT DATA YANG TELAH DIOLAH MENJADI DATA BERSIH

In [16]:
df.to_csv('bersih_most_active.csv', index=False)